In [0]:
# ============================================================
# Banking Data Pipeline
# Notebook : 01_Ingestion
# Layer    : Bronze
# Purpose  : Ingest Banking Transaction Files
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
from uuid import uuid4

# ============================================================
# Configuration
# ============================================================

BASE_PATH = "/Volumes/banking_catalog/banking_project/banking_volume"

INCOMING_PATH = f"{BASE_PATH}/incoming"
PROCESSED_PATH = f"{BASE_PATH}/processed"

BRONZE_PATH = f"{BASE_PATH}/bronze/transactions"

PIPELINE_NAME = "Banking Transaction Pipeline"

PIPELINE_RUN_ID = str(uuid4())

print("="*80)
print(f"Pipeline Run Id : {PIPELINE_RUN_ID}")
print("="*80)

In [0]:
transaction_schema = StructType([

    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("account_number", StringType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("branch", StringType(), True),
    StructField("transaction_timestamp", TimestampType(), True)

])

In [0]:
SOURCE_SYSTEMS = sorted([

    folder.name.replace("/", "")

    for folder in dbutils.fs.ls(INCOMING_PATH)

])

print("Source Systems")

for source in SOURCE_SYSTEMS:

    print(source)

In [0]:

spark.sql(f"""

CREATE TABLE IF NOT EXISTS banking_catalog.banking_project.bronze_transactions

USING DELTA

LOCATION '{BRONZE_PATH}'

""")

In [0]:
total_files = 0
total_records = 0
processed_sources = 0

for source in SOURCE_SYSTEMS:

    print("\n" + "="*80)
    print(f"Processing Source : {source}")
    print("="*80)

    source_path = f"{INCOMING_PATH}/{source}"

    processed_path = f"{PROCESSED_PATH}/{source}"

    try:

        files = dbutils.fs.ls(source_path)

        csv_files = [

            file

            for file in files

            if file.path.lower().endswith(".csv")

        ]

        if len(csv_files) == 0:

            print("No CSV Files Found")

            continue

        csv_paths = [

            file.path

            for file in csv_files

        ]

        df = (

            spark.read

            .option("header","true")

            .schema(transaction_schema)

            .csv(csv_paths)

        )

        record_count = df.count()

        total_records += record_count

        total_files += len(csv_files)

        bronze_df = (

            df

            .withColumn("source_system",lit(source))

            .withColumn("source_file",input_file_name())

            .withColumn("pipeline_name",lit(PIPELINE_NAME))

            .withColumn("pipeline_run_id",lit(PIPELINE_RUN_ID))

            .withColumn("load_date",current_date())

            .withColumn("load_timestamp",current_timestamp())

        )

        (

            bronze_df

            .write

            .format("delta")

            .mode("append")

            .save(BRONZE_PATH)

        )

        print(f"Records Written : {record_count}")

        dbutils.fs.mkdirs(processed_path)

        for file in csv_files:

            destination = f"{processed_path}/{file.name}"

            dbutils.fs.mv(file.path,destination)

            print(f"Moved : {file.name}")

        processed_sources += 1

        print("Status : SUCCESS")

    except Exception as ex:

        print(f"Status : FAILED")

        print(str(ex))

In [0]:
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

display(bronze_df.limit(20))

In [0]:
print("="*80)

print("PIPELINE SUMMARY")

print("="*80)

print(f"Pipeline Name      : {PIPELINE_NAME}")

print(f"Pipeline Run Id    : {PIPELINE_RUN_ID}")

print(f"Sources Processed  : {processed_sources}")

print(f"Files Processed    : {total_files}")

print(f"Records Loaded     : {total_records}")

print(f"Load Time          : {datetime.now()}")

print("="*80)

print("Bronze Ingestion Completed Successfully")

print("="*80)